# OWOD — eloszlás-tudatos aktív annotáció és inkrementális tanulás

**Mit mérünk.** A detektor 19 osztályt ismer. A világban 80 van. Adott egy annotációs
keret — mondjuk körönként 600 régió, amit egy ember felcímkéz. **Melyik régiókat kérjük?**
És amikor az új osztály ismertté válik, **mit adunk vissza a régiekből**, hogy ne felejtsen?

A pontszám, amit a kutatási terv javasol, és amit a 2026-08-25-i konzultáció három ponton
átdefiniált:

$$ s(x) \;=\; U(x) \;+\; \lambda\,D(x) \;+\; \gamma\,w(\hat c(x))\cdot \mathrm{coh}(x) $$

| tag | mit mér | a konzultáció után |
|---|---|---|
| $U$ | bizonytalanság — a poszterior entrópiája | **változatlan** |
| $D$ | diverzitás | nem egy fix horgonytól, hanem a **növekvő címkézett halmaztól** való távolság, plusz **batch-diverzitás** |
| $w$ | ritkaság | a jelölt klaszterének mérete — **ugyanabból a klaszterezésből**, amiből $D$ is jön |
| $\mathrm{coh}$ | lokális támogatottság | **bináris kapu**: 0 vagy 1, nem súly |

**Két üzemmód.** `RUN_GPU = False` mellett a notebook a commitolt PROB-átfutáson dolgozik:
80 000 valódi jelöltrégió, GPU és adathalmaz nélkül, pár perc. Ez válaszolja meg,
**mit választ ki egy pontszám**. `RUN_GPU = True` mellett a PROB súlyai ténylegesen
frissülnek, és a lánc végigmegy a task1 → task10 soron. Ez válaszolja meg, **mennyit
felejt és mennyit tanul**.

> **A két üzemmód nem cserélhető fel.** A korábbi munka lemérte: a fagyasztott
> jellemzőkön alapuló szimuláció a felejtés szerint **fordított sorrendbe** rakja az
> akvizíciós módszereket, mint a valódi detektor. Tehát a szimuláció összemérheti, mit
> *választ ki* egy pontszám, de nem állíthatja, hogy egyik arm kevesebbet felejt.

## Paraméterek — csak ezt kell átírni

In [ ]:
# ============================== PARAMETERS ==============================
RUN_GPU = False          # False: frozen pool, CPU, minutes. True: real PROB training.
SMOKE_TEST = True        # GPU only: two short tasks first, to prove the pipeline works
                         # before committing hours. Set False for the real chain.

# --- the task chain ---------------------------------------------------
N_TASKS               = 10     # task 1 is PROB's pretrained t1.pth; 9 more follow
BUDGET_PER_TASK       = 600    # regions the annotator is asked about, per task
ROUNDS_PER_TASK       = 6      # 1 = 600x1, 6 = 6x100, 12 = 12x50  (consultation, 7)
CANDIDATE_IMAGES      = 4000   # unlabelled images offered to the selector each task
PROPOSALS_PER_IMAGE   = 50     # PROB offers 100. 4000 x 50 proposals is a 400 MB
                               # export; 4000 x 100 is 800 MB and Colab notices.

# --- the four experimental variables ----------------------------------
ARM                   = "prior_consult_batch"   # see owl.selection.ARMS
LABELLING_POLICY      = "known_plus_selected"   # box_only | full_image | known_plus_selected
REPLAY_ARM            = "tail_favouring"        # none | uniform | head_favouring | tail_favouring
REPLAY_REALLOCATE     = False  # True: re-size the memory every task instead of growing it

# --- training (GPU only) ----------------------------------------------
EPOCHS                = 5
LEARNING_RATE         = 2e-4   # PROB's own default. 2e-5 was the earlier bottleneck.
BATCH_SIZE            = 2
EVAL_MAX_PER_CLASS    = 150    # caps evaluation cost; see section 8
EVAL_REMAINDER_RATIO  = 1
TIME_BUDGET_MINUTES   = 420    # stop cleanly before Colab does it for you

# --- the smoke test overrides, applied only when SMOKE_TEST is on ------
SMOKE = dict(N_TASKS=3, BUDGET_PER_TASK=100, ROUNDS_PER_TASK=2,
             CANDIDATE_IMAGES=300, PROPOSALS_PER_IMAGE=50, EPOCHS=1,
             EVAL_MAX_PER_CLASS=8, EVAL_REMAINDER_RATIO=0, TIME_BUDGET_MINUTES=45)

# --- reproducibility ---------------------------------------------------
SEED                  = 0
N_CLUSTERS            = 1600

# --- GPU paths ---------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/OWL"   # t1.pth and the annotation archives live here
# ========================================================================

# The smoke test only shrinks the GPU chain. The CPU sections cost minutes either
# way, so there is nothing to shrink there and nothing to be gained by doing it.
if RUN_GPU and SMOKE_TEST:
    globals().update(SMOKE)
    print("SMOKE_TEST is on: two short tasks, a tiny evaluation split. This proves the\n"
          "pipeline runs end to end. It does NOT produce a result worth reporting —\n"
          "set SMOKE_TEST = False once it has passed.")
    print({key: globals()[key] for key in SMOKE})
elif RUN_GPU:
    print("SMOKE_TEST is off: the full chain. Expect roughly\n"
          f"  {N_TASKS - 1} tasks x (train + evaluate) — see the cost print below.")

## Környezet

In [ ]:
import json, os, subprocess, sys, tarfile, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "owl").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if RUN_GPU:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/owod-active")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/gubiczam/owod-active.git", str(ROOT)], check=True)
    else:
        subprocess.run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=ROOT, check=True)
        subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)

sys.path.insert(0, str(ROOT))
import numpy as np
from owl import (bridge, clustering, evaluation_subset, labelling, metrics,
                 proposals, protocol, replay, runner, scoring, selection)
from owl.runner import table

print("owl from:", ROOT)
print("mode:", "GPU — PROB weights are updated" if RUN_GPU else "CPU — frozen PROB pass")

## 1. A taszk-lánc: egy új osztály taszkonként

Ez a legfontosabb szerkezeti döntés, és a konzultáció kérte így.

A korábbi felállásban egy inkrementális lépés **húsz** új osztályt adott hozzá 600
annotációból — osztályonként ~30 régió. Az eredmény mérhetetlen volt: az új osztályok
mAP50-je **0,010** lett, miközben a teljes t2-felügyelet 36,13-at ér el. A csereárfolyam
(hány régi mAP-pontot fizetünk egy új pontért) **2931** volt a teljes felügyelet 0,20-a
ellenében.

Taszkonként **egy** osztállyal ugyanaz a 600 régió mind egy osztályra megy. Ez az a
változtatás, ami a plaszticitást egyáltalán mérhetővé teszi.

Az osztálysorrend nem szabad: a PROB kiértékelője pozíció szerint indexeli az osztályokat,
tehát csak a hivatalos sorrend prefixét lehet ismertté nyilvánítani. Szerencsére ez a
prefix magától lefedi mind a három gyakorisági csoportot.

In [ ]:
chain = protocol.build_chain(N_TASKS)
print(table(protocol.describe_chain(chain)))

## 2. A jelöltkészlet

`RUN_GPU = False` esetén ez a commitolt PROB-átfutás: az `exps/SOWODB/PROB/t1.pth`
checkpoint egyetlen forward-passe 2 400 benchmark-képen, minden régió a benchmark saját
annotációjához illesztve IoU 0,5-nél. Ez teszi lehetővé, hogy az egész annotációs kör
laptopon fusson.

Amit egy jelölt hordoz: doboz, 256 dimenziós PROB-dekóder-embedding, osztály-poszterior,
objectness. **A kiválasztás ezeken kívül semmit nem lát** — az `oracle()` hívás az egyetlen
hely, ahol címke előkerül, és azt csak a `labelling` modul hívja meg, a keret elköltése
*után*.

In [ ]:
pool = proposals.from_frozen_pool(split="pool")
print(table([pool.describe()]))

oracle = pool.oracle()
groups = protocol.load_groups()
group_of = np.asarray([groups.get(name, "") for name in oracle.class_name])
unknown = oracle.kind == "unknown"
print("\nreal unknown objects in the pool, by frequency group:")
print(table([{g: int(((group_of == g) & unknown).sum()) for g in ("head", "medium", "tail")}]))
print(f"\nonly {unknown.mean():.1%} of proposals sit on a real unknown object — "
      "that is the needle the score has to find.")

## 3. Egy klaszterezés, amiből $D$ és $w$ is kijön

*(konzultáció, 3. pont)*

Ne külön becsüljük a ritkaságot és a diverzitást. Particionáljuk a jelöltkészletet
**egyszer** az embedding-térben, és olvassuk le mind a kettőt ugyanarról a partícióról:

- **ritkaság** = milyen kicsi a jelölt klasztere;
- **diverzitás** = milyen messze van a klasztere a known-tartalmú klaszterektől.

A partíció minőségi mérőszáma nem sziluett-érték, hanem **known-szennyezés**: hány már
ismert elem esik olyan klaszterbe, amit unknown-jelöltnek minősítenénk.

**Ez a diagnosztika orákulum nélkül lefuttatható**, mert a detektor a saját ismert
osztályait maga is fel tudja címkézni — és utólag ellenőrizhető a benchmark annotációi
ellen. A két oszlop különbsége mondja meg, mennyire bízhatunk a becslésben.

Egy részlet, ami könnyen elrontja: „a klaszter többsége known" szabály itt degenerált,
mert a készlet 81%-a háttér, tehát szinte egy klaszter sem ér el 50% knownt. A helyes
kérdés az **dúsulás**: több known van ebben a klaszterben, mint egy véletlenszerűben?

In [ ]:
detector_known = clustering.predicted_known(pool.posterior, len(protocol.TASK1))
truth_known = pool.oracle().kind == "known"
print(f"the detector labels its own known classes with precision "
      f"{float((truth_known & detector_known).sum() / detector_known.sum()):.2f} — "
      "no annotation involved")

rows = []
for k in (200, 400, 800, 1600, 3200):
    part = clustering.fit(pool.embeddings, n_clusters=k, seed=SEED)
    rows.append({
        "n_clusters": k,
        "mean_cluster_size": float(part.sizes[part.sizes > 0].mean()),
        "contamination_estimated": clustering.contamination(part, detector_known)["contamination"],
        "contamination_verified": clustering.contamination(part, truth_known)["contamination"],
        "unknown_recall": clustering.contamination(part, truth_known)["unknown_recall"],
    })
print("\n" + table(rows, digits=3))

partition = clustering.fit(pool.embeddings, n_clusters=N_CLUSTERS, seed=SEED)
print(f"\nusing K = {N_CLUSTERS} for everything below.")

## 4. A koherencia-kapu: bináris DBSCAN

*(konzultáció, 2. pont)*

A kérés: `coh(x) ∈ {0, 1}` — kapcsoló, ne súly. Ha egy jelölt DBSCAN-zajpont, akkor
`coh = 0`, azt nem akarjuk megtanulni.

Az indoklás jó: egy magányos, semmire nem hasonlító régió nagy eséllyel rossz doboz.
**A mérés viszont nem támasztja alá — és a mérés fontosabb.** A kapu a PROB saját
jellemzőterében *gyakrabban* dobja ki a valódi ismeretlen objektumokat, mint a hátteret.

A mechanizmus nem rejtélyes: a készlet 81%-a háttér, a háttérrégiók pedig szinte
egymás másolatai, tehát ők ülnek a tér legsűrűbb részén. Egy sűrű környezetben lenni ebben
a készletben azt jelenti, hogy **hátérnek nézel ki**. A táblázat utolsó oszlopa a lényeg:
a kapu után az ismeretlen objektumok aránya *csökken*.

**Amit ebből tanulunk, és amit a 3. pont old meg:** ha ugyanabból a klaszterezésből jön a
ritkaság is, akkor a hatalmas háttér-klaszter magától ~0 súlyt kap — a `w · coh` szorzat
tehát a *ritkaság* oldalán szűri ki a hátteret, nem a kapu oldalán.

In [ ]:
rows = []
for eps in (0.15, 0.25, 0.35, 0.45):
    part = clustering.fit(pool.embeddings, method="dbscan", eps=eps,
                          min_samples=5, pca_dimensions=32, seed=SEED)
    noise = part.is_noise
    kinds = pool.oracle().kind
    rows.append({
        "eps": eps,
        "clusters": part.n_clusters,
        "noise_background": float(noise[kinds == "background"].mean()),
        "noise_known": float(noise[kinds == "known"].mean()),
        "noise_real_unknown": float(noise[kinds == "unknown"].mean()),
        "unknown_share_before": float((kinds == "unknown").mean()),
        "unknown_share_after_gate": float((kinds[~noise] == "unknown").mean()),
    })
print(table(rows, digits=3))
print("\nRead the last two columns: the gate lowers the share of real unknown objects\n"
      "at every eps. It removes what we are trying to find.")

## 5. A kiválasztás: melyik pontszám mit vesz meg

*(konzultáció, 1. és 7. pont)*

Minden arm **ugyanazt a 600 régiót** kérdezi meg — az orákulum ára azonos. A kérdés az,
hogy mi jön vissza a pénzért. A `rounds` oszlop a 7. pont: egy lépésben adjuk oda a
keretet, vagy 100-asával, közben újraszámolva.

Az armok, és hogy mi az egy változó, amiben eltérnek:

| arm | mi ez |
|---|---|
| `random` | a padló |
| `entropy` | csak $U$ — a klasszikus aktív tanulás |
| `objectness` | **az ingyenes kontroll**: `objectness × √terület`, semmi tanulás, semmi eloszlás |
| `plan` | a kutatási terv egyenlete pontosan úgy, ahogy le van írva |
| `consult` | a konzultáció $D$-je (címkézettektől mért távolság) + bináris kapu |
| `consult_batch` | ugyanaz + batch-diverzitás |
| `prior_consult` | az ingyenes prior **szorzóként**, a konzultáció tagjaival |
| `prior_consult_batch` | ugyanaz + batch-diverzitás — a szintézis |

In [ ]:
rows = []
for name in ("random", "entropy", "objectness", "plan",
             "consult", "consult_batch", "prior_consult", "prior_consult_batch"):
    for rounds in (1, ROUNDS_PER_TASK):
        picked = selection.select(pool, selection.ARMS[name], budget=BUDGET_PER_TASK,
                                  rounds=rounds, n_known=len(protocol.TASK1),
                                  partition=partition)
        index = picked.indices
        found = pool.oracle().kind[index] == "unknown"
        g = group_of[index][found]
        names = pool.oracle().class_name[index][found]
        rows.append({
            "arm": name, "rounds": rounds,
            "unknown_objects": int(found.sum()),
            "head": int((g == "head").sum()),
            "medium": int((g == "medium").sum()),
            "tail": int((g == "tail").sum()),
            "classes": int(np.unique(names).size),
            "tail_classes": int(np.unique(names[g == "tail"]).size),
            "images": int(picked.images(pool).size),
        })
print(table(sorted(rows, key=lambda r: -r["tail"])))

### Mit olvasunk ki ebből

Három dolog, és mind a három ellenőrizhető a fenti táblázatban.

1. **A terv egyenlete önmagában nem veri a randomot.** A `plan` sor az egyenlet pontos
   implementációja, és nagyjából a random szintjén találja meg az ismeretleneket. Ez nem
   implementációs hiba: a `U + λD + γw·coh` három tagja együtt éppen az izolált
   outliereket kedveli, ahogy a terv maga is figyelmeztet rá.

2. **A konzultáció javításai működnek.** `consult` és `consult_batch` a terv verziójának
   a többszörösét találja meg, és a `rounds` növelése *csak* azokon az armokon segít,
   amelyeknek van mit frissíteniük — az `objectness` sor változatlan 1 és 6 kör között,
   a `consult` nem. Ez a 7. pont, lemérve.

3. **A tail oszlop a lényeg, nem az összesen.** Az `objectness` a legtöbb ismeretlent
   találja, de főleg nagy, feltűnő head-objektumokat. A `prior_consult_batch` kevesebb
   ismeretlent talál összesen, viszont **több tail-objektumot** — a keretet a fejről a
   farokra tolja át. Ez pontosan az az állítás, amit a kutatási terv tesz.

## 6. Kép vagy régió: mit címkézünk valójában

*(konzultáció, 5. pont — „ez torzít el minden más mérést, ha rosszul dől el")*

A kiválasztás egy **régióra** mutat. Az annotátor egy **képet** kap. Egy képen négy doboz
is lehet. Három szabály:

- **`box_only`** — csak a kiválasztott doboz kap címkét. A képen minden más **háttérként**
  tanít, beleértve a valódi objektumokat is.
- **`full_image`** — a képen minden annotált objektum megkapja a címkéjét. Nincs
  félcímkézés, viszont képenként több annotációs egységbe kerül.
- **`known_plus_selected`** — a **known** objektumok ingyen vannak (a detektor már tudja
  őket, nem kell hozzá ember), a kiválasztott unknown megkapja a címkét, a többi unknown
  **ignore** lesz, nem háttér.

A `half_labelled_share` oszlop az, amiről ez az egész szól: a háttérként tanított
régiók hány százaléka ül valójában egy valódi annotált objektumon.

In [ ]:
picked = selection.select(pool, selection.ARMS[ARM], budget=BUDGET_PER_TASK,
                          rounds=ROUNDS_PER_TASK, n_known=len(protocol.TASK1),
                          partition=partition)
rows = []
for policy in labelling.POLICIES:
    annotation = labelling.annotate(pool, picked, policy=policy,
                                    known_classes=protocol.TASK1)
    rows.append(annotation.summary() | {
        "half_labelled_share": labelling.half_labelling_rate(annotation, pool),
        "cost_vs_budget": annotation.oracle_cost / BUDGET_PER_TASK,
        "supervision_per_oracle_unit": annotation.labelled.size / annotation.oracle_cost,
    })
print(table(rows, digits=3))

### A válasz

`known_plus_selected` **ugyanannyiba kerül, mint a `box_only`** — mert a known objektumok
felcímkézéséhez nem kell ember —, **nulla félcímkézéssel**, és **négy-hatszor annyi
felügyeletet** ad a tanításnak ugyanazért az árért.

Ez egybevág azzal, amit a korábbi GPU-futás már megmutatott: amikor a képen amúgy meglévő
task-1 annotációkat visszatettük, a felejtés **27 pontról 2,7-re** esett — replay nélkül.
A memória nem hiányzott; eldobtuk azt, ami ott volt.

> **A GPU-oldali leképezés.** A PROB tanító-loaderében ez a `--supervision-mode`
> kapcsoló: `ft` megtartja az előző taszkok dobozait a kiválasztott képeken,
> `train` eldobja őket. A `box_only` a `train`, a másik kettő az `ft`. Egy valódi
> doboz-szintű szabályhoz szűrt XML-eket kellene írni; ez a következő lépés, nem
> ennek a futásnak a része.

## 7. Replay: mit adunk vissza a régiekből

*(konzultáció, 4. pont — a kutatási terv B kontribúciója)*

$$ m_c \;\propto\; n_c^{\alpha}, \qquad \sum_c m_c = M $$

- $\alpha = 0$: osztályonként egyenlő — a mai standard;
- $\alpha = 1$: mérettel arányos — head-favorizáló;
- $\alpha < 0$: tail-favorizáló.

Három külön kísérleti tengely, és itt három külön paraméter: a memória **mérete**, a
**szabály** ($\alpha$), és hogy taszkonként **újraosztjuk-e** vagy visszük tovább.

Az alábbi tábla a t1 tizenkilenc osztályára mutatja, mit jelent ez konkrétan — a
legritkább osztály 1 294 objektumból tanult, a leggyakoribb 262 465-ből.

In [ ]:
counts = protocol.load_train_counts()
task1_sizes = {name: counts[name] for name in protocol.TASK1}
order = sorted(task1_sizes, key=task1_sizes.get)
rows = []
for name, spec in replay.ARMS.items():
    if spec["total"] == 0:
        continue
    allocation = replay.allocate(task1_sizes, total=spec["total"], alpha=spec["alpha"])
    rows.append({
        "arm": name, "alpha": spec["alpha"], "allocated": sum(allocation.values()),
        "rarest (motorbike-scale)": allocation.get(order[0], 0),
        "median class": allocation.get(order[len(order) // 2], 0),
        "commonest (person)": allocation.get(order[-1], 0),
    })
print(table(rows))
print("\nAt alpha = 1 the rarest class gets a single exemplar and the commonest gets\n"
      "hundreds. That is the failure the research plan predicts for head-favouring\n"
      "allocation, and `minimum=1` is what stops it from reaching zero.")

## 8. A szimulált lánc — mit vesz meg a keret taszkonként

Ez még mindig a fagyasztott készleten fut, tehát **detekciós metrikát nem ad**. Azt
mutatja meg, hogyan alakul a kiválasztás összetétele, ahogy a címkézett halmaz nő és a
$D$ tag egyre nehezebben talál újat.

In [ ]:
config = runner.CycleConfig(
    n_tasks=N_TASKS, budget_per_task=BUDGET_PER_TASK, rounds_per_task=ROUNDS_PER_TASK,
    arm=ARM, labelling_policy=LABELLING_POLICY, replay_arm=REPLAY_ARM,
    replay_reallocate=REPLAY_REALLOCATE, n_clusters=N_CLUSTERS, seed=SEED,
)
results = runner.simulate(pool, config, chain=chain, partition=partition)
keep = ["task", "new_class", "asked", "objects", "head", "medium", "tail",
        "classes_seen", "label_labelled", "label_oracle_cost", "label_half_labelled"]
print(table([{k: r.flat()[k] for k in keep} for r in results], digits=3))

## 9. A valódi lánc GPU-n

Innentől a PROB súlyai ténylegesen frissülnek, és a PROB saját kiértékelője ad számot.
`RUN_GPU = False` mellett ez a rész kihagyódik.

**Amire szükség van a Drive-on** (`DRIVE_ROOT`):

```
OWL/
  checkpoints/SOWODB/t1.pth              a PROB publikált t1 checkpointja (478 MB)
  data/owdetr_pool_annotations.tar.gz    a jelöltképek VOC-XML annotációi
  data/owdetr_test_annotations.tar.gz    a teszt-annotációk (a repóban is benne van)
```

A képek nem kellenek a Drive-ra: a notebook a COCO-ról tölti le őket igény szerint,
csak azokat, amiket a kiválasztás megnyit.

**Költség.** A kiértékelés a drága rész, nem a tanítás — a teljes 4 952 képes teszt
checkpointonként ~32 perc. Ezért a lánc egy **közös, csökkentett** teszthalmazon mér,
ami minden deklarált osztályból megtartja a képeket (osztályonként legfeljebb
`EVAL_MAX_PER_CLASS`-t), és determinisztikusan mintavételez maradékot. Az így kapott
previous-class mAP **mintabecslés**, publikált teljes-teszt számokkal nem
összehasonlítható — armok között viszont igen, mert mind ugyanazon a halmazon fut.

Minden hívás **újraindítható**: ha a kimenet már megvan, a hívás kimarad. Egy megszakadt
Colab-session ott folytatja, ahol abbahagyta.

### Előellenőrzés — mi van meg a Drive-on, mi nincs

**Ezt futtasd le először,** mielőtt bármi hosszút indítanál. Megnézi, hol vannak a
szükséges fájlok — több szokásos helyen keres, a `DAOWOD` mappát is beleértve, ha korábbi
futásokból már ott van valami —, és kiírja, mi hiányzik. Semmit nem tölt le és nem tanít.

Ha egy sor `MISSING`, az alatta lévő magyarázat megmondja, honnan kell odatenni.

In [ ]:
if not RUN_GPU:
    print("RUN_GPU = False — nothing to check. Set it True and rerun this cell.")
    DRIVE = None
else:
    from google.colab import drive as _drive
    _drive.mount("/content/drive", force_remount=False)

    MY = Path("/content/drive/MyDrive")
    # look in the new folder first, then in the one earlier runs used
    SEARCH_ROOTS = [Path(DRIVE_ROOT), MY / "OWL", MY / "DAOWOD"]

    def locate(*relative_names):
        """First existing match for any of these relative paths, in any root."""
        for root in SEARCH_ROOTS:
            for name in relative_names:
                candidate = root / name
                if candidate.exists():
                    return candidate
        return None

    # the checkpoint is 478 MB and has to be uploaded once; the annotations ship
    # with the repository, so only fall back to Drive if the clone is incomplete
    CHECKPOINT = locate("checkpoints/SOWODB/t1.pth", "checkpoints/t1.pth", "t1.pth")

    STAGING = ROOT / "data" / "staging"
    POOL_ANNOTATIONS = (
        STAGING / "owdetr_pool_annotations.tar.gz"
        if (STAGING / "owdetr_pool_annotations.tar.gz").exists()
        else locate("data/owdetr_pool_annotations.tar.gz", "data/daowod_pool.tar.gz")
    )
    TEST_ANNOTATIONS = STAGING / "owdetr_test_annotations.tar.gz"

    checks = [
        ("PROB t1 checkpoint (478 MB)", CHECKPOINT,
         "Copy exps/SOWODB/PROB/t1.pth from the AI_SSD to "
         f"{DRIVE_ROOT}/checkpoints/SOWODB/t1.pth — this is the only upload needed."),
        ("candidate annotations", POOL_ANNOTATIONS,
         "Ships with the repository. If it is missing, rebuild it on the laptop with "
         "tools/build_pool_annotations.py and upload it to " f"{DRIVE_ROOT}/data/"),
        ("test annotations", TEST_ANNOTATIONS,
         "Ships with the repository — if this is missing the clone is broken."),
    ]
    print(f"{'what':34s} {'status':8s} where")
    print("-" * 100)
    missing = []
    for label, path, remedy in checks:
        if path is None or not Path(path).exists():
            print(f"{label:34s} {'MISSING':8s} → {remedy}")
            missing.append(label)
        else:
            size = path.stat().st_size / 1e6
            print(f"{label:34s} {'ok':8s} {path}  ({size:.0f} MB)")

    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True,
                         check=False)
    print("-" * 100)
    print("GPU:", gpu.stdout.strip() or "NONE — pick a GPU runtime under Runtime > Change runtime type")

    if missing:
        raise SystemExit(f"Cannot start: {missing}. Fix the lines marked MISSING above.")
    print("\nEverything is in place. Run the next cells.")

In [ ]:
if not RUN_GPU:
    print("RUN_GPU = False — the GPU chain is skipped.\n"
          "Everything above ran on the committed PROB pass and is complete on its own.")
    gpu_results = []
else:
    import shutil
    assert shutil.which("nvidia-smi"), "Select a GPU runtime first."
    assert DRIVE is not None, "Run the preflight cell above first."

    DRIVE = Path(DRIVE_ROOT)
    DATA = Path("/content/data/OWOD")
    WORK = DRIVE / "work"
    WORK.mkdir(parents=True, exist_ok=True)
    (DATA / "ImageSets" / "OWDETR").mkdir(parents=True, exist_ok=True)

    PROB = bridge.ensure_checkout(Path("/content/PROB"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROB / "requirements.txt")], check=False)

    # Deformable-DETR's multi-scale attention is a compiled CUDA extension. PROB
    # falls back to a pure-PyTorch implementation when it is absent, which is
    # correct but several times slower — at chain length nine that is the
    # difference between one evening and three. Build it once per runtime; the
    # build takes a few minutes and is cached under /content.
    try:
        import MultiScaleDeformableAttention  # noqa: F401
        print("MultiScaleDeformableAttention: already built")
    except ImportError:
        print("building MultiScaleDeformableAttention (a few minutes) ...")
        build = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-v", "."],
            cwd=PROB / "models" / "ops", capture_output=True, text=True, check=False,
        )
        try:
            import MultiScaleDeformableAttention  # noqa: F401
            print("MultiScaleDeformableAttention: built")
        except ImportError:
            print("MultiScaleDeformableAttention: BUILD FAILED — PROB will use the\n"
                  "  pure-PyTorch fallback. Correct but slower; budget more time.\n"
                  "  last lines of the build log:")
            print("\n".join(build.stdout.strip().splitlines()[-8:]))

    # the preflight cell already found these; extract both into one Annotations/ tree
    for source in (POOL_ANNOTATIONS, TEST_ANNOTATIONS):
        with tarfile.open(source) as handle:
            handle.extractall(DATA)
    print("annotations extracted:", len(list((DATA / "Annotations").glob("*.xml"))))

    declared = [t.new_class for t in chain[1:]]
    subset = evaluation_subset.from_archive(
        ROOT / "data" / "staging" / "owdetr_test_annotations.tar.gz",
        declared, seed=SEED, remainder_multiplier=EVAL_REMAINDER_RATIO,
        max_per_class=EVAL_MAX_PER_CLASS,
    )
    TEST_SET = "owl_shared_eval"
    evaluation_subset.write_image_set(
        DATA / "ImageSets" / "OWDETR" / f"{TEST_SET}.txt", subset)
    print(f"shared evaluation split: {len(subset.image_ids)} images, "
          f"~{len(subset.image_ids) * 32 / 4952:.0f} min per checkpoint")
    print("declared-class objects in play:", dict(subset.object_counts))

    candidate_index = json.loads(
        (ROOT / "data" / "reference" / "per_image_class_counts.json").read_text())
    print(f"candidate pool: {len(candidate_index)} unlabelled images")

    prob_bridge = bridge.Bridge(prob_root=PROB, data_root=DATA,
                                log_dir=WORK / "logs", num_workers=2)
    print(prob_bridge.check())

### Képek behúzása

Csak azok a képek kellenek, amiket a kiválasztás megnyit, plusz a közös teszthalmaz.
A COCO nyilvános URL-jeiről töltjük őket, párhuzamosan.

In [ ]:
if RUN_GPU:
    from concurrent.futures import ThreadPoolExecutor

    JPEG = DATA / "JPEGImages"
    JPEG.mkdir(exist_ok=True)

    def fetch(image_id, split):
        target = JPEG / f"{image_id}.jpg"
        if target.exists():
            return
        url = f"http://images.cocodataset.org/{split}/{image_id}.jpg"
        subprocess.run(["curl", "-sfL", "-o", str(target), url], check=False)

    with ThreadPoolExecutor(max_workers=16) as pool_exec:
        list(pool_exec.map(lambda i: fetch(i, "val2017"), subset.image_ids))
    print("test images ready:", sum(1 for i in subset.image_ids if (JPEG / f"{i}.jpg").exists()),
          "of", len(subset.image_ids))

In [ ]:
if RUN_GPU:
    gpu_config = runner.CycleConfig(
        n_tasks=N_TASKS, budget_per_task=BUDGET_PER_TASK, rounds_per_task=ROUNDS_PER_TASK,
        candidate_images_per_task=CANDIDATE_IMAGES,
        proposals_per_image=PROPOSALS_PER_IMAGE, arm=ARM,
        labelling_policy=LABELLING_POLICY, replay_arm=REPLAY_ARM,
        replay_reallocate=REPLAY_REALLOCATE, epochs=EPOCHS,
        learning_rate=LEARNING_RATE, batch_size=BATCH_SIZE,
        n_clusters=N_CLUSTERS, seed=SEED,
    )
    gpu_results = runner.run_chain(
        prob_bridge, gpu_config,
        workspace=WORK / ARM,
        candidate_index=candidate_index,
        start_checkpoint=CHECKPOINT,
        test_set=TEST_SET, chain=chain,
        time_budget_minutes=TIME_BUDGET_MINUTES,
    )
    print("\ncost:", prob_bridge.cost_report())

## 10. Az eredménytáblázat

Taszkonként hat szám. Az utolsó a legfontosabb: a **csereárfolyam** — hány régi mAP-pontot
fizetünk egy új mAP-pontért. A teljes t2-felügyelet **0,20**-at fizet. Ha egy futás 70-et
fizet, akkor az nem csere, hanem veszteség.

In [ ]:
if gpu_results:
    keep = ["task", "new_class", "known_mAP50", "prev_mAP50", "new_mAP50",
            "U_Recall50", "forgetting", "exchange_rate", "images_opened",
            "target_objects_in_images"]
    print(table([{k: r.flat().get(k) for k in keep} for r in gpu_results]))
else:
    print("No GPU results in this session. Reference points from earlier measured runs:\n")
    root = ROOT / "data" / "reference" / "measured"
    reference = []
    for label, filename, baseline in (
        ("full t2 supervision (ceiling)", "full_t2_supervision_metrics.json", 73.649),
        ("random, 600 regions",           "random_b600_metrics.json",         73.649),
        ("distribution-aware, 600",       "mult_prior_shrunk_b600_metrics.json", 73.649),
        ("objectness prior, 600",         "objectness_prior_b600_metrics.json",  73.649),
    ):
        path = root / filename
        if not path.exists():
            continue
        evaluation = metrics.from_bridge_metrics(path)
        row = metrics.task_row(evaluation, task="t2", new_class=None,
                               previous_baseline=baseline)
        row = {"run": label} | row
        row["exchange_rate"] = metrics.exchange_rate(row)
        reference.append(row)
    print(table(reference, digits=3))
    print("\nThese are the twenty-classes-at-once runs. The exchange rate is what the\n"
          "one-class-per-task chain above exists to fix.")

## Mit lehet és mit nem lehet állítani ebből

**Lehet.**
- Melyik pontszám mennyi valódi ismeretlent és mennyi *tail*-objektumot vesz meg azonos
  orákulum-költségen. Ez a fagyasztott készleten mérhető, minden arm ugyanazon a
  geometrián fut.
- Hogy a körökre bontás csak azoknak az armoknak segít, amelyeknek van mit frissíteniük.
- Hogy a három címkézési szabály mennyibe kerül, és mennyi félcímkézést okoz.
- Hogy a known-szennyezés orákulum nélkül becsülhető, és mennyire pontosan.

**Nem lehet.**
- Hogy egyik arm kevesebbet *felejt*, mint a másik — ez csak a valódi detektoron mérhető,
  és a fagyasztott szimuláció ezt bizonyítottan **fordítva** rangsorolja.
- Publikált PROB-számokkal összevetni a csökkentett teszthalmazon mért mAP-ot.
- Szignifikanciát három magból. Három mag előjelpróbát bír el, p-értéket nem.

**A hatókör.** Egy benchmark (S-OWODB), egy alapmodell (PROB), és a fagyasztott gerinc
miatt a dobozok soha nem javulnak: az unknown-lefedettség felülről korlátos azzal, amit
`t1.pth` egyáltalán javasol (a jelölthalmaz ismeretlen objektumainak 13,4%-a).